# FINOVA v4.1 — Evaluación financiera de proyectos

Versión refinada para una evaluación financiera más completa, trazable y reutilizable.

## Estructura del modelo
1. Proyecto y contexto estratégico.
2. Flujos operativos.
3. Financiamiento y amortización.
4. Costo de capital: **CAPM / Ke** y **WACC**.
5. VAN, TIR, payback y capacidad de pago (DSCR).
6. Escenarios por cambios en flujos y **tasa de descuento**.
7. Presentación ejecutiva automática.

### Cambio metodológico importante
Cuando se utiliza WACC, el VAN del proyecto se calcula con flujos operativos **antes del servicio de deuda**, porque el costo de la deuda ya está incorporado en la tasa de descuento. La deuda se analiza por separado mediante amortización y cobertura del servicio (DSCR), evitando contar dos veces el costo financiero.


## 1. Motor financiero auditable en Python
Estas funciones documentan de forma explícita las ecuaciones que utiliza FINOVA.


In [ ]:
# FINOVA v4.1 — motor financiero auditable en Python
# La aplicación web replica estas ecuaciones en JavaScript para funcionar sin servidor.

def cuota_nivelada(principal, tasa_nominal_anual, pagos_por_ano, anos):
    n = int(round(pagos_por_ano * anos))
    if principal <= 0:
        return 0.0
    i = tasa_nominal_anual / pagos_por_ano
    if abs(i) < 1e-12:
        return principal / n
    return principal * i / (1 - (1 + i) ** (-n))

def tasa_efectiva_anual(tasa_nominal_anual, pagos_por_ano):
    return (1 + tasa_nominal_anual / pagos_por_ano) ** pagos_por_ano - 1

def capm(tasa_libre_riesgo, beta, prima_mercado, prima_riesgo_pais=0.0):
    """Ke = rf + beta*ERP + CRP"""
    return tasa_libre_riesgo + beta * prima_mercado + prima_riesgo_pais

def wacc(ke, kd_efectiva, peso_deuda, tasa_impuesto=0.0):
    """WACC = Ke*E + Kd*(1-T)*D; E = 1-D."""
    d = peso_deuda
    e = 1 - d
    return ke * e + kd_efectiva * (1 - tasa_impuesto) * d

def tabla_amortizacion(principal, tasa_nominal_anual, pagos_por_ano, anos):
    pago = cuota_nivelada(principal, tasa_nominal_anual, pagos_por_ano, anos)
    saldo = principal
    filas = []
    n = int(round(pagos_por_ano * anos))
    for periodo in range(1, n + 1):
        interes = saldo * (tasa_nominal_anual / pagos_por_ano)
        capital = pago - interes
        if periodo == n or capital > saldo:
            capital = saldo
            pago_real = capital + interes
        else:
            pago_real = pago
        saldo = max(0.0, saldo - capital)
        filas.append({
            "periodo": periodo,
            "pago": pago_real,
            "interes": interes,
            "capital": capital,
            "saldo": saldo
        })
    return filas

def van(tasa_descuento, inversion, flujos):
    return -inversion + sum(
        flujo / (1 + tasa_descuento) ** t
        for t, flujo in enumerate(flujos, start=1)
    )

def tir_biseccion(inversion, flujos, limite_inferior=-0.9999, limite_superior=10.0, iteraciones=180):
    serie = [-inversion] + list(flujos)
    def vpn(r):
        return sum(f / (1 + r) ** i for i, f in enumerate(serie))
    lo, hi = limite_inferior, limite_superior
    flo, fhi = vpn(lo), vpn(hi)
    if flo * fhi > 0:
        return None
    for _ in range(iteraciones):
        mid = (lo + hi) / 2
        fm = vpn(mid)
        if abs(fm) < 1e-8:
            return mid
        if flo * fm <= 0:
            hi = mid
        else:
            lo, flo = mid, fm
    return (lo + hi) / 2

def payback_descontado(inversion, flujos, tasa_descuento):
    acumulado = 0.0
    for t, flujo in enumerate(flujos, start=1):
        vp = flujo / (1 + tasa_descuento) ** t
        if acumulado + vp >= inversion:
            restante = inversion - acumulado
            return (t - 1) + restante / vp if vp else None
        acumulado += vp
    return None

def dscr(flujo_operativo, servicio_deuda):
    return None if servicio_deuda <= 0 else flujo_operativo / servicio_deuda

print("Motor financiero de auditoría cargado: cuota, tasa efectiva, CAPM, WACC, amortización, VAN, TIR, payback y DSCR.")


## 2. Autoverificación
Pruebas simples para comprobar que las fórmulas fundamentales responden como se espera.


In [ ]:
# Autoverificación de coherencia matemática (sin cargar un caso de usuario)
assert round(capm(0.045, 1.20, 0.055, 0.028), 4) == 0.1390
assert abs(tasa_efectiva_anual(0.12, 12) - ((1.01)**12 - 1)) < 1e-12
assert cuota_nivelada(0, 0.10, 12, 5) == 0
print("✓ Pruebas básicas superadas.")


## 3. Generador de la aplicación
La interfaz funciona en HTML, CSS y JavaScript puro. El notebook genera un único archivo HTML autónomo que puede abrirse localmente o publicarse como sitio estático.


In [ ]:
from pathlib import Path
from IPython.display import IFrame, display, Markdown

HTML_APP = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>FINOVA · Evaluación financiera de proyectos</title>
<style>
:root{
  --navy:#061541;--navy2:#0d2c78;--blue:#234fa8;--blue2:#17367f;
  --cyan:#18b8d3;--cyan2:#91e9f3;--yellow:#ffd51f;
  --bg:#f2f5fa;--card:#fff;--text:#17213c;--muted:#6b7690;--line:#dfe6f2;
  --green:#0d9d70;--green-bg:#e5f7ef;--red:#d2444b;--red-bg:#ffeded;
  --orange:#d67c08;--orange-bg:#fff4df;--purple:#7656d6;--shadow:0 16px 44px rgba(10,31,77,.09);
}
*{box-sizing:border-box}html{scroll-behavior:smooth}
body{margin:0;background:var(--bg);color:var(--text);font-family:Inter,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif}
button,input,select,textarea{font:inherit}button{cursor:pointer}
.shell{display:grid;grid-template-columns:292px 1fr;min-height:100vh}
.sidebar{background:linear-gradient(180deg,#06143f 0%,#0d2d79 70%,#07558f 100%);color:#fff;padding:24px 18px;position:sticky;top:0;height:100vh;overflow:auto}
.brand{display:flex;align-items:center;gap:12px;padding:6px 9px 22px}.logo{width:47px;height:47px;border-radius:15px;background:linear-gradient(135deg,var(--cyan),#8eeaf5);display:grid;place-items:center;color:#071746;font-weight:950;box-shadow:0 10px 30px rgba(24,184,211,.25)}.brand b{font-size:21px}.brand small{display:block;color:#bfd0f3;margin-top:2px}
.side-title{font-size:10px;color:#afbee4;text-transform:uppercase;letter-spacing:.12em;font-weight:950;padding:5px 10px 10px}
.steps{display:grid;gap:6px}.step{width:100%;border:0;background:transparent;color:#dce7ff;text-align:left;padding:11px 12px;border-radius:13px;display:flex;align-items:center;gap:11px}.step:hover,.step.active{background:rgba(255,255,255,.12);color:#fff}.step .num{width:29px;height:29px;border-radius:9px;display:grid;place-items:center;background:rgba(255,255,255,.09);font-size:11px;font-weight:950}.step.active .num{background:var(--yellow);color:#071746}.step.done .num{background:#27ca9b;color:#052c22}
.side-note{margin:22px 8px 0;padding:14px;border:1px solid rgba(255,255,255,.15);border-radius:14px;color:#ccd9f5;font-size:12px;line-height:1.55}
.main{min-width:0}.topbar{height:76px;background:rgba(255,255,255,.94);backdrop-filter:blur(12px);border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;padding:0 28px;position:sticky;top:0;z-index:30}.topbar small{display:block;font-size:10px;text-transform:uppercase;letter-spacing:.1em;color:var(--muted);font-weight:900}.topbar strong{display:block;font-size:18px;margin-top:2px;max-width:640px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.toolbar{display:flex;gap:8px}
.btn{border:1px solid var(--line);background:#fff;color:var(--text);padding:10px 13px;border-radius:11px;font-weight:850}.btn:hover{transform:translateY(-1px)}.btn.primary{background:var(--blue);border-color:var(--blue);color:#fff}.btn.soft{background:#ecfbfe;border-color:#c8eff4;color:#07677b}
.content{max-width:1540px;margin:auto;padding:28px}.panel{display:none}.panel.active{display:block}
.page-head{display:flex;align-items:flex-end;justify-content:space-between;gap:18px;margin-bottom:18px}.page-head h1{margin:0;font-size:30px}.page-head p{margin:5px 0 0;color:var(--muted);line-height:1.5}.badge{display:inline-flex;padding:7px 10px;border-radius:999px;background:#edf3ff;color:#2a4e99;font-size:12px;font-weight:900;white-space:nowrap}
.card{background:var(--card);border:1px solid var(--line);border-radius:21px;padding:22px;box-shadow:var(--shadow)}.card h2,.card h3{margin:0 0 7px}.lead{color:var(--muted);line-height:1.55;margin:0 0 18px}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:18px}.grid3{display:grid;grid-template-columns:repeat(3,1fr);gap:18px}.grid4{display:grid;grid-template-columns:repeat(4,1fr);gap:14px}.grid5{display:grid;grid-template-columns:repeat(5,1fr);gap:12px}
.form-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:15px}.field label{display:block;font-size:12px;font-weight:850;color:#5c6880;margin-bottom:6px}.field .hint{font-size:11px;color:#8893a8;margin-top:5px;line-height:1.38}
input,select,textarea{width:100%;border:1px solid #ccd7e7;border-radius:11px;padding:11px 12px;background:#fbfcff;color:var(--text);outline:none}textarea{min-height:88px;resize:vertical}input:focus,select:focus,textarea:focus{border-color:var(--cyan);box-shadow:0 0 0 3px rgba(24,184,211,.10)}
.action-row{display:flex;justify-content:flex-end;gap:10px;margin-top:18px}.callout{padding:14px 15px;border-radius:13px;background:#eefafe;border-left:4px solid var(--cyan);color:#34435f;line-height:1.55}.callout.good{background:var(--green-bg);border-left-color:var(--green)}.callout.bad{background:var(--red-bg);border-left-color:var(--red)}.callout.warn{background:var(--orange-bg);border-left-color:var(--orange)}
.table-wrap{overflow:auto;border:1px solid var(--line);border-radius:14px}table{width:100%;border-collapse:collapse;font-size:13px}th{background:#f5f8fc;color:#53617b;text-align:right;text-transform:uppercase;letter-spacing:.045em;font-size:11px;padding:11px 10px;border-bottom:1px solid var(--line);white-space:nowrap}th:first-child,td:first-child{text-align:left}td{padding:11px 10px;border-bottom:1px solid #edf1f7;text-align:right;white-space:nowrap}tr:last-child td{border-bottom:0}.row-input{width:140px;text-align:right;padding:8px 9px}
.kpi{background:#fff;border:1px solid var(--line);border-radius:18px;padding:18px;box-shadow:0 8px 24px rgba(16,35,78,.05)}.kpi small{color:var(--muted);font-weight:850}.kpi .value{font-size:25px;font-weight:950;margin-top:8px}.kpi .sub{font-size:11px;color:var(--muted);margin-top:5px}
.hero{background:radial-gradient(circle at 88% 12%,rgba(24,184,211,.24),transparent 30%),linear-gradient(135deg,#061542,#153b98 70%,#1757ba);color:#fff;border-radius:28px;padding:30px;box-shadow:var(--shadow);position:relative;overflow:hidden}.hero:after{content:"";position:absolute;width:260px;height:260px;border:42px solid rgba(255,213,31,.07);border-radius:50%;right:-100px;bottom:-170px}.hero-inner{position:relative;z-index:1}.hero .eyebrow{font-size:11px;text-transform:uppercase;letter-spacing:.13em;color:#85e7f3;font-weight:950}.hero h1{font-size:40px;line-height:1.06;margin:10px 0 10px}.hero p{color:#dce8ff;line-height:1.55;margin:0;max-width:950px}.verdict-grid{display:grid;grid-template-columns:1.2fr .8fr;gap:15px;margin-top:20px}.glass{background:rgba(255,255,255,.10);border:1px solid rgba(255,255,255,.18);border-radius:18px;padding:18px}.glass small{font-size:10px;text-transform:uppercase;letter-spacing:.1em;color:#c9d8f6;font-weight:900}.glass .big{font-size:30px;font-weight:950;margin:7px 0}.glass p{font-size:13px;color:#e1eaff}
.donut{width:184px;height:184px;border-radius:50%;display:grid;place-items:center;position:relative;margin:4px auto;background:conic-gradient(var(--cyan) 0 0%,#e1e8f3 0 100%)}.donut:after{content:"";position:absolute;width:126px;height:126px;border-radius:50%;background:#fff}.donut .center{position:relative;z-index:2;text-align:center}.donut strong{display:block;font-size:25px}.donut span{font-size:11px;color:var(--muted)}
.stack-list{display:grid;gap:9px}.stack-row{display:grid;grid-template-columns:70px 1fr 105px;gap:10px;align-items:center;font-size:12px}.stack-bar{height:12px;background:#edf1f7;border-radius:999px;overflow:hidden;display:flex}.stack-bar .interest{background:#f0a328}.stack-bar .principal{background:#2c65c9}
.legend{display:flex;gap:16px;align-items:center;font-size:11px;color:var(--muted);margin-top:10px;flex-wrap:wrap}.legend i{display:inline-block;width:10px;height:10px;border-radius:3px;margin-right:5px}
.capm-strip{display:grid;grid-template-columns:1fr 30px 1fr 30px 1fr 30px 1fr;align-items:center;gap:6px}.capm-term{border:1px solid var(--line);border-radius:14px;padding:15px;text-align:center;background:#f9fbfe}.capm-term small{display:block;color:var(--muted);font-weight:800}.capm-term strong{display:block;font-size:22px;margin-top:7px}.op{font-size:22px;text-align:center;color:#8290a9;font-weight:900}
.formula{background:#091942;color:#eaf2ff;border-radius:14px;padding:15px 17px;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13px;line-height:1.75;overflow:auto}
.capital-bar{height:20px;border-radius:999px;overflow:hidden;background:#e5ebf4;display:flex}.capital-bar .equity{background:linear-gradient(90deg,#18b8d3,#5dd5e7)}.capital-bar .debt{background:linear-gradient(90deg,#315fc2,#173781)}
.flow-chart{height:310px;border:1px solid var(--line);border-radius:16px;padding:20px 14px 40px;position:relative;overflow:hidden;background:linear-gradient(180deg,#fbfdff,#f7f9fc)}.flow-axis{position:absolute;left:14px;right:14px;top:50%;height:1px;background:#cfd8e6}.bar-group{position:absolute;top:18px;bottom:35px;display:flex;gap:6px;align-items:center;justify-content:center}.chart-bar{position:absolute;width:22px;left:calc(50% - 25px);border-radius:7px 7px 2px 2px;background:linear-gradient(180deg,#1bc3db,#108aa5)}.chart-bar.pv{left:calc(50% + 4px);background:linear-gradient(180deg,#315fc2,#173781)}.chart-label{position:absolute;bottom:-27px;width:100%;text-align:center;font-size:11px;color:var(--muted);font-weight:850}
.pressures{display:grid;gap:10px}.pressure{border:1px solid var(--line);border-radius:14px;padding:14px}.pressure-top{display:flex;justify-content:space-between;gap:12px;font-weight:850}.pressure small{display:block;color:var(--muted);margin-top:4px}.meter{height:8px;background:#edf1f6;border-radius:99px;overflow:hidden;margin-top:10px}.meter span{display:block;height:100%;background:linear-gradient(90deg,var(--cyan),var(--blue));border-radius:99px}
.risk-list{display:grid;gap:9px}.risk{display:flex;justify-content:space-between;gap:15px;align-items:center;padding:12px 13px;border:1px solid var(--line);border-radius:12px}.risk b{font-size:13px}.risk small{display:block;color:var(--muted);margin-top:3px}.pill{padding:6px 9px;border-radius:999px;font-size:11px;font-weight:900}.pill.good{background:var(--green-bg);color:#087451}.pill.warn{background:var(--orange-bg);color:#8a5509}.pill.bad{background:var(--red-bg);color:#a82d33}
.sens-wrap{overflow:auto}.sens-grid{display:grid;grid-template-columns:120px repeat(5,minmax(110px,1fr));gap:5px;min-width:720px}.sens{padding:12px 8px;border-radius:9px;text-align:center;font-size:11px;font-weight:900}.sens.head{background:#edf2f8;color:#63718a}.sens.pos{background:#e4f7ef;color:#087551}.sens.neg{background:#ffebeb;color:#ad3030}.sens.mid{background:#fff4df;color:#8e5709}
.recs{display:grid;gap:10px}.rec{display:grid;grid-template-columns:34px 1fr;gap:10px;border:1px solid var(--line);border-radius:13px;padding:13px}.rec .n{width:30px;height:30px;border-radius:9px;background:#edf3ff;color:#284f9e;display:grid;place-items:center;font-weight:950}.rec p{margin:0;color:#40506c;line-height:1.5}
.flowline{display:grid;grid-template-columns:repeat(5,1fr);gap:8px}.flowline div{background:#f7f9fd;border:1px solid #e6ecf5;border-radius:13px;padding:12px;text-align:center;color:#5b6780;font-size:12px}.flowline b{display:block;color:#17213c;margin-bottom:4px}
.scenario-result{border-radius:18px;padding:22px;background:linear-gradient(135deg,#071846,#173c92);color:#fff}.scenario-result small{color:#b9cbed;text-transform:uppercase;letter-spacing:.09em;font-weight:900}.scenario-result .big{font-size:36px;font-weight:950;margin-top:8px}.scenario-result p{color:#dce8ff;line-height:1.5}
.presentation{background:#061642;color:#fff;min-height:660px;border-radius:25px;overflow:hidden;position:relative;box-shadow:var(--shadow)}.slide{display:none;min-height:660px;padding:56px}.slide.active{display:flex;flex-direction:column;justify-content:center}.slide .kicker{font-size:11px;text-transform:uppercase;letter-spacing:.13em;color:#7de3f1;font-weight:950}.slide h2{font-size:46px;line-height:1.07;margin:12px 0 18px}.slide p,.slide li{font-size:18px;line-height:1.55;color:#dbe8ff;max-width:1050px}.slide .huge{font-size:60px;font-weight:950}.slide-controls{position:absolute;left:24px;right:24px;bottom:24px;display:flex;justify-content:space-between;align-items:center}.slide-controls button{border:1px solid rgba(255,255,255,.24);background:rgba(255,255,255,.08);color:#fff;padding:10px 14px;border-radius:10px}.slide-counter{font-size:12px;color:#a9bde7}
.hidden{display:none!important}.footer{font-size:11px;color:#7b879d;line-height:1.55;margin:25px 2px 6px}
@media(max-width:1120px){.shell{grid-template-columns:1fr}.sidebar{position:relative;height:auto}.steps{grid-template-columns:repeat(4,1fr)}.side-note{display:none}.grid2,.grid3,.grid4,.grid5,.verdict-grid{grid-template-columns:1fr}.form-grid{grid-template-columns:1fr 1fr}.flowline{grid-template-columns:1fr}.capm-strip{grid-template-columns:1fr}.op{display:none}}
@media(max-width:700px){.content{padding:15px}.topbar{padding:0 14px}.toolbar .desktop{display:none}.steps{grid-template-columns:1fr 1fr}.form-grid{grid-template-columns:1fr}.hero h1{font-size:31px}.slide{padding:34px 24px}.slide h2{font-size:34px}.slide p,.slide li{font-size:16px}}
@media print{.sidebar,.topbar,.slide-controls{display:none!important}.shell{display:block}.content{padding:0}.panel{display:block!important;page-break-after:always}.presentation{min-height:0;background:#fff;color:#111}.slide{display:block!important;min-height:0;page-break-after:always}.slide p,.slide li{color:#222}.slide h2,.slide .huge{color:#111}}
</style>
</head>
<body>
<div class="shell">
<aside class="sidebar">
  <div class="brand"><div class="logo">FV</div><div><b>FINOVA</b><small>Evaluación financiera de proyectos</small></div></div>
  <div class="side-title">Proceso de análisis</div>
  <div class="steps">
    <button class="step active" data-panel="proyecto"><span class="num">1</span><span>Proyecto</span></button>
    <button class="step" data-panel="operacion"><span class="num">2</span><span>Flujos operativos</span></button>
    <button class="step" data-panel="financiamiento"><span class="num">3</span><span>Financiamiento</span></button>
    <button class="step" data-panel="capital"><span class="num">4</span><span>Costo de capital</span></button>
    <button class="step" data-panel="evaluacion"><span class="num">5</span><span>Evaluación</span></button>
    <button class="step" data-panel="escenarios"><span class="num">6</span><span>Escenarios</span></button>
    <button class="step" data-panel="presentacion"><span class="num">7</span><span>Presentación</span></button>
  </div>
  <div class="side-note"><b>FINOVA 4.1</b><br>La decisión financiera y la decisión de financiamiento se analizan por separado: CAPM/Ke para capital propio, WACC para estructuras mixtas y capacidad de pago para la deuda.<br><br>La aplicación inicia vacía y funciona con cualquier proyecto.</div>
</aside>

<main class="main">
<header class="topbar">
  <div><small>FINOVA · CAPM · WACC · VAN</small><strong id="topName">Nuevo proyecto</strong></div>
  <div class="toolbar">
    <button class="btn desktop" id="openBtn">Abrir</button>
    <button class="btn desktop" id="saveBtn">Guardar</button>
    <button class="btn soft" id="newBtn">Nuevo</button>
    <button class="btn" onclick="window.print()">PDF</button>
  </div>
</header>

<div class="content">

<section class="panel active" id="proyecto">
  <div class="page-head"><div><h1>1. Proyecto</h1><p>Defina qué se desea evaluar, dónde opera y cuáles son los supuestos estratégicos básicos.</p></div><span class="badge">Contexto</span></div>
  <div class="card">
    <div class="form-grid">
      <div class="field"><label>Nombre del proyecto *</label><input id="projectName" placeholder="Nombre del proyecto"></div>
      <div class="field"><label>Analista, equipo u organización *</label><input id="analyst" placeholder="Responsable del análisis"></div>
      <div class="field"><label>Tipo de proyecto *</label><select id="projectType"><option value="">Seleccione</option><option>Nuevo negocio</option><option>Expansión de capacidad / bodega</option><option>Nuevo producto o servicio</option><option>Nueva ubicación / mercado</option><option>Compra de vehículos o equipo</option><option>Proyecto dentro de empresa existente</option><option>Otro</option></select></div>
      <div class="field"><label>País / mercado *</label><input id="country" placeholder="Ej. Costa Rica"></div>
      <div class="field"><label>Sector / industria *</label><input id="sector" placeholder="Ej. hotelería, comercio, tecnología"></div>
      <div class="field"><label>Moneda *</label><select id="currency"><option value="CRC">₡ Colón costarricense</option><option value="USD">$ Dólar estadounidense</option><option value="EUR">€ Euro</option></select></div>
      <div class="field" style="grid-column:span 3"><label>Descripción *</label><textarea id="description" placeholder="Explique brevemente qué inversión se quiere realizar y cuál es la decisión que se necesita tomar."></textarea></div>
      <div class="field"><label>Inversión inicial *</label><input id="investment" type="number" min="0" step="1000" placeholder="Monto inicial"></div>
      <div class="field"><label>Horizonte del proyecto (años) *</label><input id="years" type="number" min="1" max="15" step="1" value="5"></div>
      <div class="field"><label>Valor residual al final</label><input id="residual" type="number" min="0" step="1000" placeholder="0"><div class="hint">Opcional: valor recuperable de activos al último año.</div></div>
      <div class="field"><label>Diferenciador difícil de replicar</label><input id="differentiator" placeholder="¿Qué hace distinto al proyecto?"></div>
      <div class="field"><label>Validación del mercado</label><select id="marketValidation"><option value="none">No realizada</option><option value="partial">Preliminar</option><option value="solid">Sólida / documentada</option></select></div>
      <div class="field"><label>Dependencia de persona clave</label><select id="keyPerson"><option value="low">Baja</option><option value="medium">Media</option><option value="high">Alta</option></select></div>
      <div class="field"><label>Intensidad competitiva</label><select id="competition"><option value="low">Baja</option><option value="medium" selected>Media</option><option value="high">Alta</option></select></div>
    </div>
    <div class="action-row"><button class="btn primary" id="next1">Continuar →</button></div>
  </div>
</section>

<section class="panel" id="operacion">
  <div class="page-head"><div><h1>2. Flujos operativos</h1><p>Proyecte ingresos y gastos. FINOVA convierte esas cifras en flujos de caja para valorar el proyecto.</p></div><span class="badge">Operación</span></div>
  <div class="card">
    <div class="table-wrap"><table><thead><tr><th>Año</th><th>Ingresos</th><th>Gastos</th><th>Flujo operativo</th><th>Margen</th></tr></thead><tbody id="opRows"></tbody></table></div>
    <div class="callout" style="margin-top:14px"><b>Flujo operativo = ingresos − gastos.</b> FINOVA asume que estos montos representan los flujos disponibles del proyecto después de sus costos operativos. Si el caso requiere impuestos, depreciación o capital de trabajo detallados, deben incorporarse en las proyecciones correspondientes.</div>
    <div class="action-row"><button class="btn" data-go="proyecto">← Volver</button><button class="btn primary" id="next2">Continuar →</button></div>
  </div>
</section>

<section class="panel" id="financiamiento">
  <div class="page-head"><div><h1>3. Financiamiento</h1><p>Defina la estructura deuda–capital y las condiciones del crédito. La deuda se analiza tanto por costo como por capacidad de pago.</p></div><span class="badge">Estructura de capital</span></div>
  <div class="grid2">
    <div class="card">
      <div class="form-grid" style="grid-template-columns:1fr 1fr">
        <div class="field"><label>Porcentaje financiado con deuda (%) *</label><input id="financePct" type="number" min="0" max="100" step="1" placeholder="0 - 100"></div>
        <div class="field"><label>Tasa nominal anual de la deuda (%) *</label><input id="loanRate" type="number" min="0" max="200" step="0.1" placeholder="Tasa bancaria"></div>
        <div class="field"><label>Plazo del crédito (años) *</label><input id="loanYears" type="number" min="1" max="30" step="1" value="5"></div>
        <div class="field"><label>Pagos por año *</label><select id="ppy"><option value="12">12 · mensuales</option><option value="4">4 · trimestrales</option><option value="2">2 · semestrales</option><option value="1">1 · anual</option></select></div>
        <div class="field"><label>Tasa impositiva aplicable (%) *</label><input id="taxRate" type="number" min="0" max="100" step="0.1" value="0"><div class="hint">Se usa para estimar el costo de deuda después del escudo fiscal.</div></div>
      </div>
      <div id="termWarning" class="callout warn hidden" style="margin-top:14px"></div>
      <div class="action-row"><button class="btn" data-go="operacion">← Volver</button><button class="btn primary" id="next3">Continuar →</button></div>
    </div>
    <div class="card">
      <h2>Estructura del financiamiento</h2><p class="lead">La deuda y el capital propio no tienen el mismo costo ni el mismo riesgo.</p>
      <div class="grid2" style="align-items:center">
        <div><div class="donut" id="financeDonut"><div class="center"><strong id="donutPct">0%</strong><span>deuda</span></div></div><div class="legend" style="justify-content:center"><span><i style="background:#18b8d3"></i>Deuda</span><span><i style="background:#e1e8f3"></i>Capital propio</span></div></div>
        <div>
          <div class="kpi"><small>Monto financiado</small><div class="value" id="loanAmount">—</div></div>
          <div class="kpi" style="margin-top:9px"><small>Aporte propio</small><div class="value" id="equityAmount">—</div></div>
          <div class="kpi" style="margin-top:9px"><small>Cuota por período</small><div class="value" id="paymentAmount">—</div></div>
          <div class="kpi" style="margin-top:9px"><small>Tasa efectiva anual de deuda</small><div class="value" id="effectiveDebtRate">—</div></div>
        </div>
      </div>
    </div>
  </div>
  <div class="card" style="margin-top:18px">
    <h2>Composición anual del crédito</h2><p class="lead">La cuota se descompone en interés y amortización de capital. FINOVA mantiene esta lectura separada del VAN para no duplicar el costo de la deuda cuando se utiliza WACC.</p>
    <div class="stack-list" id="debtBars"></div>
    <div class="legend"><span><i style="background:#f0a328"></i>Interés</span><span><i style="background:#2c65c9"></i>Capital</span></div>
  </div>
</section>

<section class="panel" id="capital">
  <div class="page-head"><div><h1>4. Costo de capital</h1><p>Determine técnicamente la tasa mínima exigida al proyecto. Si hay capital propio se estima Ke con CAPM; si hay deuda y capital se calcula WACC.</p></div><span class="badge">CAPM + WACC</span></div>
  <div class="grid2">
    <div class="card">
      <div class="field"><label>Método de tasa de descuento *</label><select id="discountMode"><option value="auto">Automático · CAPM / WACC</option><option value="manual">Tasa manual justificada</option></select></div>
      <div id="autoCapital" style="margin-top:16px">
        <div class="form-grid" style="grid-template-columns:1fr 1fr">
          <div class="field"><label>Tasa libre de riesgo rf (%) *</label><input id="rf" type="number" min="0" max="100" step="0.01" placeholder="Ej. bono soberano de referencia"></div>
          <div class="field"><label>Beta del sector / proyecto *</label><input id="beta" type="number" min="0" max="10" step="0.01" value="1"></div>
          <div class="field"><label>Prima de riesgo de mercado ERP (%) *</label><input id="erp" type="number" min="0" max="100" step="0.01" placeholder="Prima del mercado maduro"></div>
          <div class="field"><label>Prima de riesgo país CRP (%)</label><input id="crp" type="number" min="0" max="100" step="0.01" value="0"></div>
        </div>
        <div class="field" style="margin-top:15px"><label>Fuentes / fecha de los parámetros</label><textarea id="capitalSources" placeholder="Ej. Treasury 10Y, beta sectorial, prima de mercado, prima país y fecha de consulta."></textarea><div class="hint">FINOVA no descarga datos automáticamente: el usuario mantiene control sobre las fuentes y la fecha de los supuestos.</div></div>
      </div>
      <div id="manualCapital" class="hidden" style="margin-top:16px">
        <div class="field"><label>Tasa manual (%) *</label><input id="manualRate" type="number" min="0" max="200" step="0.01" placeholder="Tasa requerida"></div>
        <div class="field" style="margin-top:15px"><label>Justificación de la tasa</label><textarea id="manualReason" placeholder="Explique el costo de oportunidad, política corporativa o fuente utilizada."></textarea></div>
      </div>
      <div class="action-row"><button class="btn" data-go="financiamiento">← Volver</button><button class="btn primary" id="analyzeBtn">Calcular y evaluar →</button></div>
    </div>
    <div class="card">
      <h2 id="capitalMethodTitle">Tasa requerida calculada</h2><p class="lead" id="capitalMethodLead">Complete los parámetros para obtener el costo de capital.</p>
      <div class="grid2">
        <div class="kpi"><small>Costo de capital propio · Ke</small><div class="value" id="keValue">—</div><div class="sub">CAPM + prima país</div></div>
        <div class="kpi"><small>Costo efectivo de deuda · Kd</small><div class="value" id="kdValue">—</div><div class="sub">tasa efectiva anual</div></div>
        <div class="kpi"><small>Kd después de impuestos</small><div class="value" id="kdTaxValue">—</div><div class="sub">escudo fiscal</div></div>
        <div class="kpi"><small>Tasa de descuento seleccionada</small><div class="value" id="discountValue">—</div><div class="sub" id="discountValueSub">—</div></div>
      </div>
      <div style="margin-top:18px">
        <div class="capital-bar"><span class="equity" id="equityBar" style="width:100%"></span><span class="debt" id="debtBar" style="width:0%"></span></div>
        <div class="legend"><span><i style="background:#18b8d3"></i>Capital propio</span><span><i style="background:#244fa8"></i>Deuda</span></div>
      </div>
    </div>
  </div>
  <div class="grid2" style="margin-top:18px">
    <div class="card">
      <h2>CAPM</h2>
      <div class="formula">Ke = rf + β × ERP + CRP</div>
      <p class="lead" style="margin-top:12px">El costo de capital propio incorpora valor del dinero en el tiempo, riesgo sistemático del sector y, cuando corresponde, riesgo país.</p>
    </div>
    <div class="card">
      <h2>WACC</h2>
      <div class="formula">WACC = Ke × E/(D+E) + Kd × (1−T) × D/(D+E)</div>
      <p class="lead" style="margin-top:12px">En una estructura mixta, FINOVA pondera el costo de los accionistas y el costo de la deuda después del escudo fiscal.</p>
    </div>
  </div>
</section>

<section class="panel" id="evaluacion">
  <div class="page-head"><div><h1>5. Evaluación</h1><p>FINOVA convierte los supuestos en una decisión: creación de valor, margen de seguridad, capacidad de pago y riesgos no financieros.</p></div><span class="badge">Dashboard</span></div>
  <div id="emptyResult" class="card"><h2>Aún no hay resultados</h2><p class="lead">Complete los pasos anteriores y pulse <b>Calcular y evaluar</b>.</p><button class="btn primary" data-go="proyecto">Empezar</button></div>
  <div id="resultArea" class="hidden">
    <div class="hero"><div class="hero-inner">
      <div class="eyebrow">Resultado del análisis</div><h1 id="resultName"></h1><p id="resultDesc"></p>
      <div class="verdict-grid">
        <div class="glass"><small>Decisión financiera</small><div class="big" id="verdict"></div><p id="verdictText"></p></div>
        <div class="glass"><small>Valor Actual Neto</small><div class="big" id="heroVan"></div><p id="heroVanSub"></p></div>
      </div>
    </div></div>

    <div class="grid5" style="margin-top:16px">
      <div class="kpi"><small>VAN</small><div class="value" id="kVan">—</div><div class="sub">creación / destrucción de valor</div></div>
      <div class="kpi"><small>TIR</small><div class="value" id="kIrr">—</div><div class="sub" id="kIrrSub">—</div></div>
      <div class="kpi"><small>Tasa requerida</small><div class="value" id="kDiscount">—</div><div class="sub" id="kDiscountSub">—</div></div>
      <div class="kpi"><small>Spread TIR − tasa</small><div class="value" id="kSpread">—</div><div class="sub">margen de rentabilidad</div></div>
      <div class="kpi"><small>DSCR mínimo</small><div class="value" id="kDscr">—</div><div class="sub">capacidad de pago de deuda</div></div>
    </div>

    <div class="grid2" style="margin-top:18px">
      <div class="card">
        <h2>Flujos del proyecto y valor presente</h2><p class="lead">Turquesa: flujo operativo del proyecto. Azul: el mismo flujo descontado con Ke/WACC. El costo de la deuda ya está incorporado en WACC, por eso el servicio de deuda se analiza aparte.</p>
        <div class="flow-chart" id="flowChart"><div class="flow-axis"></div></div>
        <div class="legend"><span><i style="background:#18b8d3"></i>Flujo para VAN</span><span><i style="background:#244fa8"></i>Valor presente</span></div>
      </div>
      <div class="card">
        <h2>¿Qué presiona el resultado?</h2><p class="lead">Peso relativo de la inversión inicial y de los gastos operativos en valor presente.</p>
        <div class="pressures" id="pressureList"></div>
        <div class="callout" style="margin-top:14px" id="wealthMessage"></div>
      </div>
    </div>

    <div class="grid2" style="margin-top:18px">
      <div class="card">
        <h2>Capacidad de pago del crédito</h2><p class="lead">FINOVA compara el flujo operativo con el servicio anual de deuda. Este análisis de liquidez complementa el VAN.</p>
        <div class="table-wrap"><table><thead><tr><th>Año</th><th>Flujo operativo</th><th>Servicio deuda</th><th>Flujo después deuda</th><th>DSCR</th></tr></thead><tbody id="debtCapacityRows"></tbody></table></div>
      </div>
      <div class="card">
        <h2>Riesgos no financieros</h2><p class="lead">La viabilidad financiera no sustituye la revisión del mercado y de la ejecución.</p>
        <div class="risk-list" id="riskList"></div>
      </div>
    </div>

    <div class="grid2" style="margin-top:18px">
      <div class="card">
        <h2>Condiciones de equilibrio</h2><p class="lead">¿Cuánto margen existe antes de destruir valor?</p>
        <div class="recs" id="recommendations"></div>
      </div>
      <div class="card">
        <h2>Lectura ejecutiva</h2><div id="executive" class="callout"></div>
        <div class="flowline" style="margin-top:14px"><div><b>1 · Operación</b>Ingresos − gastos</div><div><b>2 · Capital</b>CAPM / WACC</div><div><b>3 · Descuento</b>Flujos a valor presente</div><div><b>4 · Deuda</b>Capacidad de pago</div><div><b>5 · Decisión</b>VAN + TIR + riesgos</div></div>
      </div>
    </div>

    <div class="card" style="margin-top:18px">
      <h2>Auditoría del VAN</h2><p class="lead">El VAN del proyecto se calcula con flujos operativos antes del servicio de deuda y se descuenta con la tasa que refleja la estructura de capital.</p>
      <div class="table-wrap"><table><thead><tr><th>Año</th><th>Ingresos</th><th>Gastos</th><th>Residual</th><th>Flujo para VAN</th><th>Factor VP</th><th>Valor presente</th></tr></thead><tbody id="auditRows"></tbody></table></div>
    </div>

    <div class="action-row"><button class="btn" data-go="capital">← Revisar supuestos</button><button class="btn primary" data-go="escenarios">Analizar escenarios →</button></div>
  </div>
</section>

<section class="panel" id="escenarios">
  <div class="page-head"><div><h1>6. Escenarios</h1><p>Analice el VAN bajo otras condiciones, especialmente cambios en la tasa requerida y en los flujos operativos, sin alterar el proyecto base.</p></div><span class="badge">Sensibilidad</span></div>
  <div id="scenarioEmpty" class="card"><h2>Primero analice el proyecto</h2><p class="lead">La sensibilidad parte de la última evaluación calculada.</p><button class="btn primary" data-go="evaluacion">Ir a evaluación</button></div>
  <div id="scenarioArea" class="hidden">
    <div class="grid2">
      <div class="card">
        <h2>Matriz VAN: flujo operativo × tasa</h2><p class="lead">Las filas cambian todos los flujos operativos; las columnas modifican la tasa de descuento en puntos porcentuales.</p>
        <div class="sens-wrap"><div class="sens-grid" id="scenarioGrid"></div></div>
        <div class="legend"><span><i style="background:#e4f7ef;border:1px solid #bfe9d8"></i>VAN positivo</span><span><i style="background:#fff4df;border:1px solid #eed6aa"></i>Cerca de equilibrio</span><span><i style="background:#ffebeb;border:1px solid #f2c7c7"></i>VAN negativo</span></div>
      </div>
      <div class="card">
        <h2>Escenario personalizado</h2><p class="lead">Pruebe una combinación específica sin cambiar los datos guardados.</p>
        <div class="form-grid" style="grid-template-columns:1fr 1fr 1fr">
          <div class="field"><label>Ajuste de ingresos (%)</label><input id="scRevenue" type="number" step="0.1" value="0"></div>
          <div class="field"><label>Ajuste de gastos (%)</label><input id="scExpense" type="number" step="0.1" value="0"></div>
          <div class="field"><label>Ajuste tasa (p.p.)</label><input id="scRate" type="number" step="0.1" value="0"></div>
        </div>
        <div class="scenario-result" style="margin-top:16px"><small>VAN del escenario</small><div class="big" id="scenarioVan">—</div><p id="scenarioText">Cambie las variables para explorar el resultado.</p></div>
      </div>
    </div>
    <div class="grid3" style="margin-top:18px">
      <div class="kpi"><small>Tasa base</small><div class="value" id="scenarioBaseRate">—</div><div class="sub">Ke / WACC del proyecto</div></div>
      <div class="kpi"><small>Tasa crítica</small><div class="value" id="scenarioCriticalRate">—</div><div class="sub">aprox. TIR del proyecto</div></div>
      <div class="kpi"><small>Margen de tasa</small><div class="value" id="scenarioRateMargin">—</div><div class="sub">TIR − tasa base</div></div>
    </div>
    <div class="action-row"><button class="btn" data-go="evaluacion">← Volver a evaluación</button><button class="btn primary" data-go="presentacion">Preparar presentación →</button></div>
  </div>
</section>

<section class="panel" id="presentacion">
  <div class="page-head"><div><h1>7. Presentación</h1><p>Resumen automático: contexto, flujos, financiamiento, costo de capital, resultado y conclusión.</p></div><span class="badge">Exposición</span></div>
  <div id="presEmpty" class="card"><h2>Primero analice el proyecto</h2><p class="lead">La presentación se genera con el último análisis.</p><button class="btn primary" data-go="evaluacion">Ir a evaluación</button></div>
  <div id="presentation" class="presentation hidden">
    <div class="slide active"><div class="kicker">1 · Introducción</div><h2 id="s1Title"></h2><p id="s1Text"></p></div>
    <div class="slide"><div class="kicker">2 · Proyecto</div><h2>¿Qué se está evaluando?</h2><ul id="s2List"></ul></div>
    <div class="slide"><div class="kicker">3 · Financiamiento</div><h2>Estructura de capital</h2><p id="s3Text"></p></div>
    <div class="slide"><div class="kicker">4 · Tasa requerida</div><h2 id="s4Title"></h2><div class="huge" id="s4Rate"></div><p id="s4Text"></p></div>
    <div class="slide"><div class="kicker">5 · Resultado</div><h2>Valor Actual Neto</h2><div class="huge" id="s5Van"></div><p id="s5Text"></p></div>
    <div class="slide"><div class="kicker">6 · Conclusión</div><h2 id="s6Title"></h2><ul id="s6List"></ul></div>
    <div class="slide-controls"><button id="prevSlide">← Anterior</button><span class="slide-counter" id="slideCounter">1 / 6</span><button id="nextSlide">Siguiente →</button></div>
  </div>
</section>

<div class="footer">FINOVA es una herramienta de evaluación financiera de proyectos. CAPM, WACC, VAN, TIR, capacidad de deuda y escenarios dependen de los supuestos ingresados. La aplicación no reemplaza una revisión tributaria, contable, legal, de mercado o de riesgo especializada.</div>
</div>
</main>
</div>
<input type="file" id="fileInput" accept=".json" class="hidden">

<script>
const $=id=>document.getElementById(id);
const symbols={CRC:"₡",USD:"$",EUR:"€"};
let analyzed=false,lastModel=null,lastBE=null,slideIndex=0;

function show(id){
  document.querySelectorAll(".panel").forEach(p=>p.classList.toggle("active",p.id===id));
  document.querySelectorAll(".step").forEach(s=>s.classList.toggle("active",s.dataset.panel===id));
  window.scrollTo({top:0,behavior:"smooth"});
}
document.querySelectorAll(".step").forEach(b=>b.addEventListener("click",()=>show(b.dataset.panel)));
document.querySelectorAll("[data-go]").forEach(b=>b.addEventListener("click",()=>show(b.dataset.go)));

function num(id){return Number($(id).value)||0}
function clamp(x,a,b){return Math.min(Math.max(x,a),b)}
function money(v){
  if(!isFinite(v))return "—";
  const s=symbols[$("currency").value||"CRC"];
  return `${v<0?"−":""}${s}${Math.abs(v).toLocaleString("es-CR",{maximumFractionDigits:0})}`;
}
function compactMoney(v){
  const s=symbols[$("currency").value||"CRC"],a=Math.abs(v),sign=v<0?"−":"";
  if(a>=1e9)return `${sign}${s}${(a/1e9).toFixed(2)}B`;
  if(a>=1e6)return `${sign}${s}${(a/1e6).toFixed(2)}M`;
  if(a>=1e3)return `${sign}${s}${(a/1e3).toFixed(0)}K`;
  return money(v);
}
function pct(v,d=2){return isFinite(v)?`${Number(v).toFixed(d)}%`:"—"}

function renderYears(preserve=true){
  const years=clamp(parseInt($("years").value)||1,1,15);$("years").value=years;
  const old=[...document.querySelectorAll("#opRows tr")].map(r=>({i:Number(r.querySelector(".income")?.value)||0,e:Number(r.querySelector(".expense")?.value)||0}));
  $("opRows").innerHTML="";
  for(let y=1;y<=years;y++){
    const v=preserve&&old[y-1]?old[y-1]:{i:0,e:0},tr=document.createElement("tr");
    tr.innerHTML=`<td><b>Año ${y}</b></td><td><input class="row-input income" type="number" min="0" step="1000" value="${v.i||""}" placeholder="0"></td><td><input class="row-input expense" type="number" min="0" step="1000" value="${v.e||""}" placeholder="0"></td><td class="net">—</td><td class="margin">—</td>`;
    $("opRows").appendChild(tr);
  }
  document.querySelectorAll("#opRows input").forEach(x=>x.addEventListener("input",changed));
  refreshPreview();
}

function readData(){
  const rows=[...document.querySelectorAll("#opRows tr")];
  return {
    name:$("projectName").value.trim(),analyst:$("analyst").value.trim(),projectType:$("projectType").value,
    country:$("country").value.trim(),sector:$("sector").value.trim(),currency:$("currency").value,
    description:$("description").value.trim(),investment:num("investment"),years:rows.length,residual:num("residual"),
    differentiator:$("differentiator").value.trim(),marketValidation:$("marketValidation").value,keyPerson:$("keyPerson").value,competition:$("competition").value,
    incomes:rows.map(r=>Number(r.querySelector(".income").value)||0),expenses:rows.map(r=>Number(r.querySelector(".expense").value)||0),
    financePct:num("financePct")/100,loanRate:num("loanRate")/100,loanYears:Math.max(1,parseInt($("loanYears").value)||1),ppy:Math.max(1,parseInt($("ppy").value)||12),taxRate:num("taxRate")/100,
    discountMode:$("discountMode").value,rf:num("rf")/100,beta:num("beta"),erp:num("erp")/100,crp:num("crp")/100,capitalSources:$("capitalSources").value.trim(),
    manualRate:num("manualRate")/100,manualReason:$("manualReason").value.trim()
  };
}
function validate(stage){
  const d=readData(),e=[];
  if(stage==="project"||stage==="all"){
    if(!d.name)e.push("nombre del proyecto");if(!d.analyst)e.push("analista/equipo");if(!d.projectType)e.push("tipo de proyecto");
    if(!d.country)e.push("país/mercado");if(!d.sector)e.push("sector");if(!d.description)e.push("descripción");if(d.investment<=0)e.push("inversión inicial");
  }
  if(stage==="operation"||stage==="all"){
    if(d.incomes.some(x=>x<=0))e.push("ingresos de todos los años");if(d.expenses.some(x=>x<0))e.push("gastos de todos los años");
  }
  if(stage==="finance"||stage==="all"){
    if(d.financePct<0||d.financePct>1)e.push("porcentaje financiado");
    if(d.financePct>0&&d.loanRate<=0)e.push("tasa de la deuda");
    if(d.taxRate<0||d.taxRate>1)e.push("tasa impositiva");
  }
  if(stage==="capital"||stage==="all"){
    if(d.discountMode==="auto"){if(d.rf<=0)e.push("tasa libre de riesgo");if(d.beta<=0)e.push("beta");if(d.erp<=0)e.push("prima de riesgo de mercado")}
    else if(d.manualRate<=0)e.push("tasa manual");
  }
  return e;
}
function missing(e){alert("Complete antes de continuar: "+[...new Set(e)].join(", ")+".");}

function payment(P,annual,ppy,years){
  const N=Math.max(1,Math.round(ppy*years));if(P<=0)return 0;const i=annual/ppy;
  return Math.abs(i)<1e-12?P/N:P*i/(1-Math.pow(1+i,-N));
}
function effectiveAnnual(nominal,ppy){return Math.pow(1+nominal/ppy,ppy)-1}
function debt(d,rateOverride=null,investmentOverride=null){
  const rate=rateOverride===null?d.loanRate:rateOverride,inv=investmentOverride===null?d.investment:investmentOverride,P=inv*d.financePct,p=payment(P,rate,d.ppy,d.loanYears),N=Math.round(d.ppy*d.loanYears);
  let bal=P,totalI=0;const yrs=[];
  for(let m=0;m<N;m++){
    const y=Math.floor(m/d.ppy);if(!yrs[y])yrs[y]={open:bal,pay:0,int:0,prin:0,close:bal};
    const it=bal*(rate/d.ppy);let pr=p-it,act=p;if(m===N-1||pr>bal){pr=bal;act=bal+it}
    bal=Math.max(0,bal-pr);yrs[y].pay+=act;yrs[y].int+=it;yrs[y].prin+=pr;yrs[y].close=bal;totalI+=it;
  }
  return {P,p,yrs,totalI,kdEffective:effectiveAnnual(rate,d.ppy)};
}
function capm(d){return d.rf+d.beta*d.erp+d.crp}
function capitalCost(d,loanRateOverride=null){
  if(d.discountMode==="manual")return {ke:null,kd:effectiveAnnual(loanRateOverride??d.loanRate,d.ppy),kdTax:null,wacc:d.manualRate,rate:d.manualRate,method:"Tasa manual"};
  const ke=capm(d),kd=effectiveAnnual(loanRateOverride??d.loanRate,d.ppy),kdTax=kd*(1-d.taxRate),D=d.financePct,E=1-D,wacc=ke*E+kdTax*D;
  return {ke,kd,kdTax,wacc,rate:D>0?wacc:ke,method:D>0?"WACC":"Ke · CAPM"};
}
function npv(rate,flows){return flows.reduce((a,x,i)=>a+x/Math.pow(1+rate,i),0)}
function irr(flows){
  let lo=-.9999,hi=10,flo=npv(lo,flows),fhi=npv(hi,flows);if(!isFinite(flo)||!isFinite(fhi)||flo*fhi>0)return null;
  for(let k=0;k<180;k++){const mid=(lo+hi)/2,f=npv(mid,flows);if(Math.abs(f)<1e-8)return mid;if(flo*f<=0){hi=mid;fhi=f}else{lo=mid;flo=f}}
  return (lo+hi)/2;
}
function discountedPayback(inv,pvs){
  let c=0;
  for(let i=0;i<pvs.length;i++){if(c+pvs[i]>=inv){const rem=inv-c;return i+(pvs[i]!==0?rem/pvs[i]:1)}c+=pvs[i]}
  return null;
}
function model(d,opt={}){
  const inv=opt.investment??d.investment,rf=opt.revenueFactor??1,ef=opt.expenseFactor??1;
  const cc=capitalCost(d,opt.loanRate??null);
  const rate=opt.discountRate??cc.rate,ds=debt(d,opt.loanRate??null,inv),rows=[];
  for(let i=0;i<d.years;i++){
    const yd=ds.yrs[i]||{pay:0,int:0,prin:0},income=d.incomes[i]*rf,expense=d.expenses[i]*ef,operating=income-expense,residual=(i===d.years-1?d.residual:0),projectFlow=operating+residual;
    const factor=1/Math.pow(1+rate,i+1),pv=projectFlow*factor,afterDebt=operating-yd.pay+(i===d.years-1?d.residual:0),dscr=yd.pay>0?operating/yd.pay:null;
    rows.push({year:i+1,income,expense,operating,residual,projectFlow,factor,pv,payment:yd.pay,interest:yd.int,principal:yd.prin,afterDebt,dscr});
  }
  const van=-inv+rows.reduce((a,r)=>a+r.pv,0),flows=[-inv,...rows.map(r=>r.projectFlow)],tir=irr(flows),payback=discountedPayback(inv,rows.map(r=>r.pv));
  const dscrs=rows.filter(r=>r.dscr!==null).map(r=>r.dscr),minDscr=dscrs.length?Math.min(...dscrs):null;
  return {d,inv,cc,ds,rows,van,irr:tir,payback,minDscr,flows,discountRate:rate};
}
function bisect(fn,lo,hi){
  let flo=fn(lo),fhi=fn(hi);if(!isFinite(flo)||!isFinite(fhi)||flo*fhi>0)return null;
  for(let k=0;k<120;k++){const mid=(lo+hi)/2,f=fn(mid);if(Math.abs(f)<1e-7)return mid;if(flo*f<=0){hi=mid;fhi=f}else{lo=mid;flo=f}}
  return (lo+hi)/2;
}
function breakEven(m){
  const d=m.d,rate=m.discountRate,pvR=m.rows.reduce((a,r)=>a+r.income*r.factor,0),pvE=m.rows.reduce((a,r)=>a+r.expense*r.factor,0),pvResidual=m.rows.reduce((a,r)=>a+r.residual*r.factor,0);
  const revFactor=pvR>0?(d.investment+pvE-pvResidual)/pvR:null,expFactor=pvE>0?(pvR+pvResidual-d.investment)/pvE:null,maxInv=m.rows.reduce((a,r)=>a+r.pv,0);
  let maxLoanRate=null;
  if(d.discountMode==="auto"&&d.financePct>0){
    const f=r=>model(d,{loanRate:r}).van;if(f(0)<0)maxLoanRate=0;else if(f(3)>0)maxLoanRate=Infinity;else maxLoanRate=bisect(f,0,3);
  }
  return {pvR,pvE,pvResidual,revFactor,expFactor,maxInv,maxLoanRate};
}
function status(m){
  if(m.van<0)return {label:"NO VIABLE EN SU FORMA ACTUAL",tone:"bad",text:"El VAN es negativo: el proyecto destruye valor frente al rendimiento mínimo exigido bajo los supuestos actuales."};
  const ratio=m.d.investment?m.van/m.d.investment:0,spread=m.irr===null?null:m.irr-m.discountRate;
  if(ratio<.05||(spread!==null&&spread<.02))return {label:"VIABLE · MARGEN ESTRECHO",tone:"warn",text:"El VAN es positivo, pero el colchón financiero es reducido. Conviene revisar sensibilidad, deuda y riesgos de ejecución antes de comprometer recursos."};
  return {label:"VIABLE · CREA VALOR",tone:"good",text:"El VAN es positivo y el proyecto supera la tasa mínima requerida bajo los supuestos ingresados."};
}

function refreshPreview(){
  [...document.querySelectorAll("#opRows tr")].forEach(r=>{const i=Number(r.querySelector(".income").value)||0,e=Number(r.querySelector(".expense").value)||0,n=i-e;r.querySelector(".net").textContent=money(n);r.querySelector(".margin").textContent=i>0?pct(100*n/i):"—"});
  const d=readData(),ds=debt(d),fp=clamp(d.financePct*100,0,100);
  $("financeDonut").style.background=`conic-gradient(var(--cyan) 0 ${fp}%,#e1e8f3 ${fp}% 100%)`;$("donutPct").textContent=`${fp.toFixed(0)}%`;
  $("loanAmount").textContent=money(ds.P);$("equityAmount").textContent=money(Math.max(0,d.investment-ds.P));$("paymentAmount").textContent=money(ds.p);$("effectiveDebtRate").textContent=pct(ds.kdEffective*100);
  renderDebtBars(ds);refreshCapitalPreview();
  const w=$("termWarning");if(d.loanYears>d.years&&d.financePct>0){w.classList.remove("hidden");w.innerHTML=`<b>Atención:</b> el crédito dura ${d.loanYears} años y el horizonte solo ${d.years}. El análisis de deuda mostrará únicamente los años incluidos en el proyecto.`}else w.classList.add("hidden");
}
function renderDebtBars(ds){
  if(!ds.yrs.length){$("debtBars").innerHTML=`<div class="callout">Ingrese los datos del crédito para visualizar su composición.</div>`;return}
  const max=Math.max(...ds.yrs.map(y=>y.pay),1);
  $("debtBars").innerHTML=ds.yrs.map((y,i)=>{const wi=100*y.int/max,wp=100*y.prin/max;return `<div class="stack-row"><b>Año ${i+1}</b><div class="stack-bar"><span class="interest" style="width:${wi}%"></span><span class="principal" style="width:${wp}%"></span></div><span>${money(y.pay)}</span></div>`}).join("");
}
function refreshCapitalPreview(){
  const d=readData(),manual=d.discountMode==="manual";$("autoCapital").classList.toggle("hidden",manual);$("manualCapital").classList.toggle("hidden",!manual);
  const cc=capitalCost(d),D=clamp(d.financePct*100,0,100),E=100-D;$("equityBar").style.width=`${E}%`;$("debtBar").style.width=`${D}%`;
  $("keValue").textContent=cc.ke===null?"No aplica":pct(cc.ke*100);$("kdValue").textContent=pct(cc.kd*100);$("kdTaxValue").textContent=cc.kdTax===null?"No aplica":pct(cc.kdTax*100);$("discountValue").textContent=pct(cc.rate*100);$("discountValueSub").textContent=cc.method;
  $("capitalMethodTitle").textContent=manual?"Tasa manual seleccionada":D>0?"WACC calculado":"Ke calculado con CAPM";
  $("capitalMethodLead").textContent=manual?"FINOVA utilizará la tasa ingresada y conservará la justificación en el proyecto.":D>0?"La tasa combina el costo del capital propio y el costo de la deuda después de impuestos.":"Sin deuda, la tasa requerida corresponde al costo del capital propio.";
}

function renderFlowChart(m){
  const el=$("flowChart");el.innerHTML=`<div class="flow-axis"></div>`;const vals=m.rows.flatMap(r=>[Math.abs(r.projectFlow),Math.abs(r.pv)]),max=Math.max(...vals,1),N=m.rows.length;
  m.rows.forEach((r,i)=>{const left=(i+.5)*(100/N),group=document.createElement("div");group.className="bar-group";group.style.left=`calc(${left}% - ${45/N}px)`;group.style.width=`${90/N}px`;
    [["projectFlow",""],["pv","pv"]].forEach(([key,cls],j)=>{const v=r[key],h=Math.max(3,Math.abs(v)/max*112),bar=document.createElement("div");bar.className=`chart-bar ${cls}`;bar.style.height=`${h}px`;bar.style.left=j===0?"calc(50% - 25px)":"calc(50% + 4px)";if(v>=0){bar.style.bottom="50%"}else{bar.style.top="50%";bar.style.borderRadius="2px 2px 7px 7px"}group.appendChild(bar)});
    const lab=document.createElement("div");lab.className="chart-label";lab.textContent=`Año ${r.year}`;group.appendChild(lab);el.appendChild(group);
  });
}
function renderPressures(m,be){
  const arr=[{name:"Inversión inicial",value:m.d.investment,desc:"Desembolso del momento 0"},{name:"Gastos operativos",value:be.pvE,desc:"Valor presente de gastos proyectados"}].sort((a,b)=>b.value-a.value),total=arr.reduce((a,x)=>a+x.value,0)||1;
  $("pressureList").innerHTML=arr.map(x=>{const s=100*x.value/total;return `<div class="pressure"><div class="pressure-top"><span>${x.name}</span><span>${s.toFixed(1)}%</span></div><small>${x.desc} · ${money(x.value)}</small><div class="meter"><span style="width:${s}%"></span></div></div>`}).join("");
  const spread=m.irr===null?null:(m.irr-m.discountRate)*100,msg=$("wealthMessage");
  if(spread===null){msg.className="callout warn";msg.innerHTML="<b>TIR no definida.</b> Los flujos no permiten obtener una raíz única."}
  else if(spread>0){msg.className="callout good";msg.innerHTML=`<b>Crea riqueza financiera.</b> La TIR supera la tasa requerida en <b>${spread.toFixed(2)} p.p.</b>`}
  else{msg.className="callout bad";msg.innerHTML=`<b>Destruye riqueza financiera.</b> La TIR está <b>${Math.abs(spread).toFixed(2)} p.p.</b> por debajo de la tasa requerida.`}
}
function renderDebtCapacity(m){
  $("debtCapacityRows").innerHTML=m.rows.map(r=>`<tr><td><b>${r.year}</b></td><td>${money(r.operating)}</td><td>${r.payment>0?money(r.payment):"—"}</td><td>${money(r.afterDebt)}</td><td>${r.dscr===null?"—":r.dscr.toFixed(2)+"x"}</td></tr>`).join("");
}
function renderRisks(d){
  const items=[];
  if(d.differentiator)items.push({name:"Diferenciador",desc:d.differentiator,tone:"good",label:"Definido"});else items.push({name:"Diferenciador",desc:"No se documentó un elemento difícil de replicar.",tone:"warn",label:"Pendiente"});
  if(d.marketValidation==="solid")items.push({name:"Validación de mercado",desc:"El proyecto reporta validación sólida/documentada.",tone:"good",label:"Sólida"});
  else if(d.marketValidation==="partial")items.push({name:"Validación de mercado",desc:"Existe validación preliminar; conviene ampliarla.",tone:"warn",label:"Preliminar"});
  else items.push({name:"Validación de mercado",desc:"No se ha documentado validación de mercado.",tone:"bad",label:"Pendiente"});
  const kp={low:["Dependencia de persona clave","La continuidad no depende fuertemente de una sola persona.","good","Baja"],medium:["Dependencia de persona clave","Existe cierta dependencia operativa que conviene mitigar.","warn","Media"],high:["Dependencia de persona clave","La operación depende fuertemente de una persona o colaborador crítico.","bad","Alta"]}[d.keyPerson];
  items.push({name:kp[0],desc:kp[1],tone:kp[2],label:kp[3]});
  const comp={low:["Competencia","La intensidad competitiva declarada es baja.","good","Baja"],medium:["Competencia","La intensidad competitiva declarada es media.","warn","Media"],high:["Competencia","La competencia declarada es alta; el precio y el diferenciador son críticos.","bad","Alta"]}[d.competition];
  items.push({name:comp[0],desc:comp[1],tone:comp[2],label:comp[3]});
  $("riskList").innerHTML=items.map(x=>`<div class="risk"><div><b>${x.name}</b><small>${x.desc}</small></div><span class="pill ${x.tone}">${x.label}</span></div>`).join("");
}
function renderRecommendations(m,be){
  const rev=(be.revFactor-1)*100,exp=(be.expFactor-1)*100,items=[];
  if(m.van<0){
    if(isFinite(rev)&&rev>0)items.push(`Los ingresos tendrían que aumentar aproximadamente <b>${rev.toFixed(2)}%</b> para llevar el VAN a cero, manteniendo lo demás constante.`);
    if(isFinite(exp)&&exp<0)items.push(`Los gastos tendrían que reducirse aproximadamente <b>${Math.abs(exp).toFixed(2)}%</b> para alcanzar el equilibrio.`);
    if(be.maxLoanRate!==null&&be.maxLoanRate!==Infinity)items.push(`Con la estructura actual, una tasa nominal de deuda cercana a <b>${pct(be.maxLoanRate*100)}</b> sería el límite financiero aproximado para VAN = 0.`);
    items.push(`La inversión compatible con los flujos actuales es aproximadamente <b>${money(be.maxInv)}</b>.`);
  }else{
    if(isFinite(rev)&&rev<0)items.push(`Los ingresos pueden caer aproximadamente <b>${Math.abs(rev).toFixed(2)}%</b> antes de que el VAN llegue a cero.`);
    if(isFinite(exp)&&exp>0)items.push(`Los gastos pueden aumentar aproximadamente <b>${exp.toFixed(2)}%</b> antes de llegar al equilibrio.`);
    if(be.maxLoanRate===Infinity)items.push(`Dentro del rango probado, el costo de la deuda no elimina el VAN positivo.`);
    else if(be.maxLoanRate!==null)items.push(`La tasa nominal de deuda podría subir aproximadamente hasta <b>${pct(be.maxLoanRate*100)}</b> antes de que el VAN llegue a cero.`);
    items.push(`La inversión inicial podría llegar aproximadamente a <b>${money(be.maxInv)}</b> antes de VAN = 0, manteniendo los flujos.`);
  }
  $("recommendations").innerHTML=items.slice(0,4).map((x,i)=>`<div class="rec"><div class="n">${i+1}</div><p>${x}</p></div>`).join("");
}
function renderAudit(m){
  $("auditRows").innerHTML=m.rows.map(r=>`<tr><td><b>${r.year}</b></td><td>${money(r.income)}</td><td>${money(r.expense)}</td><td>${r.residual?money(r.residual):"—"}</td><td>${money(r.projectFlow)}</td><td>${r.factor.toFixed(4)}</td><td><b>${money(r.pv)}</b></td></tr>`).join("");
}
function renderScenarios(m){
  $("scenarioEmpty").classList.add("hidden");$("scenarioArea").classList.remove("hidden");
  const flowShocks=[-10,-5,0,5,10],rateShocks=[-4,-2,0,2,4];
  let h=`<div class="sens head">Flujo \\ Tasa</div>`+rateShocks.map(x=>`<div class="sens head">${x>=0?"+":""}${x} p.p.</div>`).join("");
  flowShocks.forEach(fs=>{h+=`<div class="sens head">${fs>=0?"+":""}${fs}% flujo</div>`;rateShocks.forEach(rs=>{const rate=Math.max(0,m.discountRate+rs/100),flows=m.rows.map(r=>r.projectFlow*(1+fs/100)),v=-m.inv+flows.reduce((a,x,i)=>a+x/Math.pow(1+rate,i+1),0),thr=m.inv*.01,cl=v>thr?"pos":v<-thr?"neg":"mid";h+=`<div class="sens ${cl}" title="${money(v)}">${compactMoney(v)}</div>`})});
  $("scenarioGrid").innerHTML=h;$("scenarioBaseRate").textContent=pct(m.discountRate*100);$("scenarioCriticalRate").textContent=m.irr===null?"No definida":pct(m.irr*100);$("scenarioRateMargin").textContent=m.irr===null?"—":pct((m.irr-m.discountRate)*100);
  updateCustomScenario();
}
function updateCustomScenario(){
  if(!lastModel)return;
  const d=lastModel.d,rev=1+num("scRevenue")/100,exp=1+num("scExpense")/100,rate=Math.max(0,lastModel.discountRate+num("scRate")/100),m=model(d,{revenueFactor:rev,expenseFactor:exp,discountRate:rate});
  $("scenarioVan").textContent=money(m.van);
  $("scenarioText").textContent=`Ingresos ${num("scRevenue")>=0?"+":""}${num("scRevenue").toFixed(1)}%, gastos ${num("scExpense")>=0?"+":""}${num("scExpense").toFixed(1)}% y tasa ${num("scRate")>=0?"+":""}${num("scRate").toFixed(1)} p.p. respecto al caso base.`;
}
["scRevenue","scExpense","scRate"].forEach(id=>$(id).addEventListener("input",updateCustomScenario));

function renderPresentation(m,be){
  const d=m.d,st=status(m),spread=m.irr===null?null:(m.irr-m.discountRate)*100;
  $("presEmpty").classList.add("hidden");$("presentation").classList.remove("hidden");
  $("s1Title").textContent=d.name;$("s1Text").innerHTML=`${d.description}<br><br>Análisis preparado por <b>${d.analyst}</b>.`;
  $("s2List").innerHTML=[`Tipo: <b>${d.projectType}</b>.`,`Mercado y sector: <b>${d.country} · ${d.sector}</b>.`,`Inversión inicial: <b>${money(d.investment)}</b>.`,`Horizonte: <b>${d.years} años</b>.`,`Ingresos acumulados: <b>${money(d.incomes.reduce((a,b)=>a+b,0))}</b>.`,`Gastos acumulados: <b>${money(d.expenses.reduce((a,b)=>a+b,0))}</b>.`].map(x=>`<li>${x}</li>`).join("");
  $("s3Text").innerHTML=d.financePct>0?`La deuda financia <b>${(d.financePct*100).toFixed(0)}%</b> de la inversión (${money(m.ds.P)}). La tasa nominal es <b>${pct(d.loanRate*100)}</b> y la efectiva anual es <b>${pct(m.ds.kdEffective*100)}</b>. El plazo es de <b>${d.loanYears} años</b>. FINOVA analiza el servicio de deuda por separado y utiliza su costo después de impuestos dentro del WACC.`:`El proyecto se financia sin deuda externa; la tasa requerida corresponde al costo del capital propio.`;
  $("s4Title").textContent=m.cc.method;$("s4Rate").textContent=pct(m.discountRate*100);
  $("s4Text").innerHTML=d.discountMode==="manual"?`La tasa fue ingresada manualmente. ${d.manualReason?`Justificación: ${d.manualReason}`:""}`:`El costo de capital propio es <b>${pct(m.cc.ke*100)}</b>${d.financePct>0?`, el costo efectivo de la deuda es <b>${pct(m.cc.kd*100)}</b> y, después de impuestos, <b>${pct(m.cc.kdTax*100)}</b>. La combinación produce un WACC de <b>${pct(m.cc.wacc*100)}</b>.`:` calculado mediante CAPM para ${d.country} y el sector ${d.sector}.`}`;
  $("s5Van").textContent=money(m.van);$("s5Text").innerHTML=`${st.text} ${m.irr!==null?`La TIR es <b>${pct(m.irr*100)}</b>, ${spread!==null?`${Math.abs(spread).toFixed(2)} p.p. ${spread>=0?"por encima":"por debajo"} de la tasa requerida`:""}.`:""}`;
  $("s6Title").textContent=m.van>=0?"Conclusión: avanzar con control de riesgos":"Conclusión: replantear antes de invertir";
  const bullets=[`VAN = <b>${money(m.van)}</b>.`,m.irr!==null?`TIR = <b>${pct(m.irr*100)}</b>.`:"TIR no definida.",m.minDscr===null?"No existe deuda que evaluar.":`DSCR mínimo = <b>${m.minDscr.toFixed(2)}x</b>.`,d.differentiator?`Diferenciador declarado: <b>${d.differentiator}</b>.`:"Conviene definir un diferenciador difícil de replicar."];
  $("s6List").innerHTML=bullets.map(x=>`<li>${x}</li>`).join("");setSlide(0);
}
function renderResult(){
  const d=readData(),m=model(d),be=breakEven(m),st=status(m);lastModel=m;lastBE=be;analyzed=true;
  $("emptyResult").classList.add("hidden");$("resultArea").classList.remove("hidden");
  $("resultName").textContent=d.name;$("resultDesc").textContent=`${d.projectType} · ${d.country} · ${d.sector}. ${d.description}`;$("verdict").textContent=st.label;$("verdictText").textContent=st.text;$("heroVan").textContent=money(m.van);$("heroVanSub").textContent=`${m.van>=0?"Crea":"Destruye"} ${money(Math.abs(m.van))} de valor frente a la tasa mínima exigida.`;
  $("kVan").textContent=money(m.van);$("kIrr").textContent=m.irr===null?"No definida":pct(m.irr*100);$("kIrrSub").textContent=m.irr===null?"No hay una raíz única identificable.":"rentabilidad interna";$("kDiscount").textContent=pct(m.discountRate*100);$("kDiscountSub").textContent=m.cc.method;
  $("kSpread").textContent=m.irr===null?"—":pct((m.irr-m.discountRate)*100);$("kDscr").textContent=m.minDscr===null?"Sin deuda":m.minDscr.toFixed(2)+"x";
  renderFlowChart(m);renderPressures(m,be);renderDebtCapacity(m);renderRisks(d);renderRecommendations(m,be);renderAudit(m);renderScenarios(m);renderPresentation(m,be);
  const ex=$("executive"),spread=m.irr===null?null:(m.irr-m.discountRate)*100;ex.className=`callout ${st.tone}`;
  let debtTxt=m.minDscr===null?"El proyecto no tiene servicio de deuda.":m.minDscr<1?`Además, el DSCR mínimo es ${m.minDscr.toFixed(2)}x, por debajo de 1.0x: existen años en los que el flujo operativo no cubre completamente la deuda.`:m.minDscr<1.2?`El DSCR mínimo es ${m.minDscr.toFixed(2)}x: la deuda se cubre, pero con un colchón reducido.`:`El DSCR mínimo es ${m.minDscr.toFixed(2)}x y muestra cobertura del servicio de deuda.`;
  ex.innerHTML=`<b>${st.label}.</b> ${st.text}${spread!==null?` La TIR está <b>${Math.abs(spread).toFixed(2)} p.p. ${spread>=0?"por encima":"por debajo"}</b> de la tasa requerida.`:""} ${debtTxt}`;
  markDone();
}
function markDone(){
  const st=[...document.querySelectorAll(".step")];
  st[0].classList.toggle("done",validate("project").length===0);st[1].classList.toggle("done",validate("operation").length===0);st[2].classList.toggle("done",validate("finance").length===0);st[3].classList.toggle("done",validate("capital").length===0);st[4].classList.toggle("done",analyzed);st[5].classList.toggle("done",analyzed);st[6].classList.toggle("done",analyzed);
}
function changed(){
  analyzed=false;$("resultArea").classList.add("hidden");$("emptyResult").classList.remove("hidden");$("scenarioArea").classList.add("hidden");$("scenarioEmpty").classList.remove("hidden");$("presentation").classList.add("hidden");$("presEmpty").classList.remove("hidden");
  $("topName").textContent=$("projectName").value.trim()||"Nuevo proyecto";refreshPreview();markDone();
}
function setSlide(i){const s=[...document.querySelectorAll(".slide")];slideIndex=(i+s.length)%s.length;s.forEach((x,j)=>x.classList.toggle("active",j===slideIndex));$("slideCounter").textContent=`${slideIndex+1} / ${s.length}`}
$("prevSlide").addEventListener("click",()=>setSlide(slideIndex-1));$("nextSlide").addEventListener("click",()=>setSlide(slideIndex+1));

$("next1").addEventListener("click",()=>{const e=validate("project");if(e.length)return missing(e);markDone();show("operacion")});
$("next2").addEventListener("click",()=>{const e=[...validate("project"),...validate("operation")];if(e.length)return missing(e);markDone();show("financiamiento")});
$("next3").addEventListener("click",()=>{const e=[...validate("project"),...validate("operation"),...validate("finance")];if(e.length)return missing(e);markDone();show("capital")});
$("analyzeBtn").addEventListener("click",()=>{const e=validate("all");if(e.length)return missing(e);renderResult();show("evaluacion")});

["projectName","analyst","projectType","country","sector","currency","description","investment","residual","differentiator","marketValidation","keyPerson","competition","financePct","loanRate","loanYears","ppy","taxRate","discountMode","rf","beta","erp","crp","capitalSources","manualRate","manualReason"].forEach(id=>{$(id).addEventListener("input",changed);$(id).addEventListener("change",changed)});
$("years").addEventListener("change",()=>{renderYears(true);changed()});

function saveProject(){
  const d=readData(),o={version:"FINOVA-4.1",...d,financePct:d.financePct*100,loanRate:d.loanRate*100,taxRate:d.taxRate*100,rf:d.rf*100,erp:d.erp*100,crp:d.crp*100,manualRate:d.manualRate*100};
  const blob=new Blob([JSON.stringify(o,null,2)],{type:"application/json"}),a=document.createElement("a");a.href=URL.createObjectURL(blob);a.download=`FINOVA_${(d.name||"proyecto").replace(/[^a-z0-9áéíóúñ_-]+/gi,"_")}.json`;a.click();URL.revokeObjectURL(a.href);
}
$("saveBtn").addEventListener("click",saveProject);$("openBtn").addEventListener("click",()=>$("fileInput").click());
$("fileInput").addEventListener("change",ev=>{
  const f=ev.target.files[0];if(!f)return;const r=new FileReader();
  r.onload=()=>{try{const x=JSON.parse(r.result);
    $("projectName").value=x.name||"";$("analyst").value=x.analyst||"";$("projectType").value=x.projectType||"";$("country").value=x.country||"";$("sector").value=x.sector||"";$("currency").value=x.currency||"CRC";$("description").value=x.description||"";$("investment").value=x.investment||"";$("years").value=x.years||5;$("residual").value=x.residual||"";$("differentiator").value=x.differentiator||"";$("marketValidation").value=x.marketValidation||"none";$("keyPerson").value=x.keyPerson||"low";$("competition").value=x.competition||"medium";
    $("financePct").value=x.financePct??"";$("loanRate").value=x.loanRate??"";$("loanYears").value=x.loanYears||5;$("ppy").value=x.ppy||12;$("taxRate").value=x.taxRate??0;$("discountMode").value=x.discountMode||"auto";$("rf").value=x.rf??"";$("beta").value=x.beta??1;$("erp").value=x.erp??"";$("crp").value=x.crp??0;$("capitalSources").value=x.capitalSources||"";$("manualRate").value=x.manualRate??"";$("manualReason").value=x.manualReason||"";
    renderYears(false);[...document.querySelectorAll("#opRows tr")].forEach((tr,i)=>{tr.querySelector(".income").value=(x.incomes||[])[i]??"";tr.querySelector(".expense").value=(x.expenses||[])[i]??""});changed();show("proyecto");
  }catch(e){alert("No fue posible abrir este archivo FINOVA.")}};
  r.readAsText(f);ev.target.value="";
});
$("newBtn").addEventListener("click",()=>{
  if(!confirm("¿Crear un proyecto nuevo? Se limpiarán los datos actuales."))return;
  ["projectName","analyst","country","sector","description","investment","residual","differentiator","financePct","loanRate","rf","erp","capitalSources","manualRate","manualReason"].forEach(id=>$(id).value="");
  $("projectType").value="";$("currency").value="CRC";$("years").value=5;$("marketValidation").value="none";$("keyPerson").value="low";$("competition").value="medium";$("loanYears").value=5;$("ppy").value=12;$("taxRate").value=0;$("discountMode").value="auto";$("beta").value=1;$("crp").value=0;analyzed=false;renderYears(false);changed();show("proyecto");
});
document.addEventListener("keydown",e=>{if($("presentacion").classList.contains("active")){if(e.key==="ArrowRight")setSlide(slideIndex+1);if(e.key==="ArrowLeft")setSlide(slideIndex-1)}});

renderYears(false);markDone();
</script>
</body>
</html>"""

output = Path("FINOVA_v4.1_Profesional.html")
output.write_text(HTML_APP, encoding="utf-8")

display(Markdown(
    "### ✅ FINOVA v4 generado\n"
    "Incluye CAPM, WACC, escudo fiscal, VAN, TIR, capacidad de deuda, escenarios por tasa y riesgos cualitativos.\n\n"
    f"Archivo: `{output.resolve()}`"
))
display(IFrame(src=str(output), width="100%", height=980))
